# Erdem'in sesi: Piper eğitimi (DizgeBERT g2p)

`voice-me-recorder` ile aldığın **eğitim zip'ini** (`voice-me-dataset-*.zip`) ya da **yedek zip'ini** (`voice-me-backup-*.zip`) yükle; defter gerisini kendi yapar.

1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → **Run all**. Zip'i yükleme penceresi çıkınca seç (ya da 5. hücredeki `ZIP_PATH` ile Drive yolu ver).
3. Eğitim bitince (ya da durdurunca) son hücre `.onnx` + `.onnx.json` üretir ve indirir. Bunları voice-me'nin Piper ses klasörüne koy.

**İki aşamalı eğitim (önerilen):**

1. **Ana model, sıfırdan:** Higgs ile üretilen 20.000 cümlelik veri setinin zip'ini (`metadata.csv` + `wavs/*.flac`) Drive'a koy, 5. hücrede `ZIP_PATH`'e yolunu yaz, 6. hücrede `PRETRAINED_CHECKPOINT` boş kalsın. Sesler otomatik 22050 Hz'e çevrilir. Bu uzun sürer (L4'te günler); oturum kapanınca defteri tekrar çalıştır, Drive'daki `last.ckpt`'tan devam eder.
2. **Kendi sesinle ince ayar:** Ana model yeterince iyi olunca `voice-me-recorder` zip'ini ver, 6. hücrede `PRETRAINED_CHECKPOINT`'e ana modelin Drive'daki `.ckpt` yolunu yaz. Model adı zip'teki konuşmacıdan gelir, ana modelin klasörüne yazmaz.

Fonemizasyon uygulamadakiyle **birebir aynı**: voice-me'nin `turkish.rs`/`dizge.rs`/`phonemes.rs` kodunun Python portu (`dizge_phonemizer.py`) eğitimde de kullanılır, config'e `phoneme_type: "dizge"` yazılır.

Oturum kapanırsa: aynı defteri yeniden çalıştır. Çıktı Drive'a yazıldığı için `last.ckpt` bulunur ve eğitim kaldığı yerden sürer.

In [ ]:
#@title 1. GPU, Drive ve kurulum
import os, subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise SystemExit("GPU yok: Runtime → Change runtime type → T4 GPU seç, sonra yeniden çalıştır.")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

#@markdown Drive bağlanırsa kontrol noktaları orada saklanır ve oturum kapansa da eğitim devam edebilir.
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT = "/content/drive/MyDrive/colab/piper" #@param {type:"string"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = DRIVE_OUTPUT
else:
    OUTPUT_ROOT = "/content/piper-out"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

!apt-get -q update -y
!apt-get -q install -y build-essential cmake ninja-build espeak-ng aria2
REPO_URL = "https://github.com/KiON-GiON/piper1-gpl.git"
REPO_DIR = "/content/piper1-gpl"
if not os.path.exists(REPO_DIR):
    !git clone -q -b fixes {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!python -m pip install -q -e .[train]
!bash build_monotonic_align.sh
!pip install -q --upgrade gdown scikit-build protobuf==3.20.3
!python setup.py build_ext --inplace
!pip install -q onnxruntime numpy
print("Kurulum tamam.")

In [ ]:
%%writefile /content/dizge_phonemizer.py
"""Training-side port of voice-me's Turkish DizgeBERT phonemizer.

A line-for-line Python port of `crates/voice-me-tts-piper/src/{turkish,dizge,
phonemes}.rs`, so a Piper voice trained on this module's phonemes hears the
same symbols voice-me feeds it at inference (`"phoneme_type": "dizge"`).

    p = DizgePhonemizer("model.onnx", "vocab.json")
    p.phonemize("tr", "Merhaba, dünya!")   # -> [["m", ...], ...] per sentence

`phonemize(voice, text)` has the signature of piper's `EspeakPhonemizer`, so it
drops into piper's training dataset unchanged. It returns one list of
single-codepoint NFD symbols per sentence, with clause marks put back.
"""

import json
import unicodedata

LETTERS = "abcçdefgğhıijklmnoöpqrsştuüvwxyzâîû"
VOWEL_LETTERS = "aeıioöuüâîû"
FRONT_VOWEL_LETTERS = "eiöüî"
LABEL_VOWELS = "IUYaeioøœuyɑɔɛɨ"
VOICE_VOWELS = "aeɛæiɪoɔuʊyøœɯ"
STRESS = "ˈ"
UNSTRESSED = {"ve", "de", "da", "ki", "mi", "mı", "mu", "mü", "ile", "bu", "şu"}
KEPT = "aeiouyøœɔɛbdfjklmnprstvzʃʒɟɾː"

CLAUSE_MARKS = ",:;.!?"
SENTENCE_MARKS = ".!?"
SPACED_MARKS = ",:;"

VOICE_SYMBOL = {
    "I": "ɪ",
    "U": "ʊ",
    "Y": "y",
    "ɑ": "a",
    "ɨ": "ɯ",
    "ł": "ɫ",
    "ʋ": "v",
    "x": "h",
    "ç": "h",
    "ɣ": "r",  # dizge writes a word-final r as a fricative; eSpeak NG as r
    "g": "ɡ",  # LATIN SMALL LETTER SCRIPT G
    "c": "k",
    "ɱ": "m",
    "ŋ": "n",
}
DROPPED = {"ʰ", "̥", "̧"}


# ---- text ---------------------------------------------------------------


def lowercase(text):
    out = []
    for ch in text:
        if ch == "I":
            out.append("ı")
        elif ch == "İ":
            out.append("i")
        else:
            out.append(ch.lower())
    return "".join(out).replace("i̇", "i")


def _known_letter(ch):
    if ch in LETTERS:
        return ch
    base = unicodedata.normalize("NFD", ch)[:1]
    if base and base != ch and base in LETTERS:
        return base
    return None


def words(clause, max_letters):
    text = spell_numbers(lowercase(clause))
    out, current = [], []
    for ch in text:
        letter = _known_letter(ch)
        if letter is not None:
            current.append(letter)
        elif ch in ("'", "’"):
            pass  # a suffix after an apostrophe belongs to its word
        elif current:
            out.append("".join(current))
            current = []
    if current:
        out.append("".join(current))
    max_letters = max(max_letters, 1)
    pieces = []
    for word in out:
        for i in range(0, len(word), max_letters):
            pieces.append(word[i : i + max_letters])
    return pieces


# ---- numbers ------------------------------------------------------------

ONES = ["", "bir", "iki", "üç", "dört", "beş", "altı", "yedi", "sekiz", "dokuz"]
TENS = ["", "on", "yirmi", "otuz", "kırk", "elli", "altmış", "yetmiş", "seksen", "doksan"]
SCALES = ["", "bin", "milyon", "milyar", "trilyon", "katrilyon", "kentilyon"]


def _below_thousand(n, parts):
    hundreds, rest = divmod(n, 100)
    if hundreds > 1:
        parts.append(ONES[hundreds])
    if hundreds > 0:
        parts.append("yüz")
    if rest // 10 > 0:
        parts.append(TENS[rest // 10])
    if rest % 10 > 0:
        parts.append(ONES[rest % 10])


def number_words(n):
    if n == 0:
        return "sıfır"
    groups = []
    rest = n
    while rest > 0:
        groups.append(rest % 1000)
        rest //= 1000
    parts = []
    for scale in range(len(groups) - 1, -1, -1):
        group = groups[scale]
        if group == 0:
            continue
        if not (scale == 1 and group == 1):  # "bin", never "bir bin"
            _below_thousand(group, parts)
        if scale > 0:
            parts.append(SCALES[scale])
    return " ".join(parts)


def spell_numbers(text):
    chars = list(text)

    def digit_at(i):
        return i < len(chars) and chars[i] in "0123456789"

    out = []
    index = 0
    while index < len(chars):
        if not digit_at(index):
            out.append(chars[index])
            index += 1
            continue
        digits = ""
        while digit_at(index):
            digits += chars[index]
            index += 1
        if len(digits) <= 3:
            while (
                index < len(chars)
                and chars[index] == "."
                and all(digit_at(index + o) for o in (1, 2, 3))
                and not digit_at(index + 4)
            ):
                digits += "".join(chars[index + 1 : index + 4])
                index += 4
        out.append(" ")
        if len(digits) <= 18 and (len(digits) == 1 or not digits.startswith("0")):
            out.append(number_words(int(digits)))
        else:
            out.append(" ".join(number_words(int(d)) for d in digits))
        if index < len(chars) and chars[index] == "," and digit_at(index + 1):
            out.append(" virgül")
            index += 1
        if not (index < len(chars) and chars[index] in ("'", "’")):
            out.append(" ")
    return "".join(out)


# ---- labels -------------------------------------------------------------


def _is_label_vowel(symbol):
    return symbol in LABEL_VOWELS


def _is_label_consonant(symbol):
    return not _is_label_vowel(symbol) and symbol not in ("ː", "ʰ", "̥")


def labels_line_up(word, labels):
    if len(word) != len(labels):
        return False
    for letter, label in zip(word, labels):
        if not label:
            return False
        first = label[0]
        if letter == "ğ":
            ok = first in ("ː", "j", "ɣ") or _is_label_vowel(first)
        elif letter == "y":
            ok = first in ("j", "I", "ː")
        elif letter in VOWEL_LETTERS:
            ok = _is_label_vowel(first)
        else:
            ok = any(_is_label_consonant(s) for s in label)
        if not ok:
            return False
    return True


def _stress(word, phonemes):
    if word in UNSTRESSED:
        return phonemes
    for at in range(len(phonemes) - 1, -1, -1):
        if phonemes[at] in VOICE_VOWELS:
            return phonemes[:at] + STRESS + phonemes[at:]
    return phonemes


def to_voice_symbols(word, labels):
    out = []
    for letter, label in zip(word, labels):
        if letter == "y" and label == "I":
            out.append("j")
            continue
        for symbol in unicodedata.normalize("NFC", label):
            if symbol in KEPT:
                out.append(symbol)
            elif symbol in DROPPED:
                continue
            elif symbol in VOICE_SYMBOL:
                out.append(VOICE_SYMBOL[symbol])
    return _stress(word, "".join(out))


def rule_phonemes(word):
    letters = list(word)

    def colouring_vowel(index):
        for letter in letters[index + 1 :] + letters[:index][::-1]:
            if letter in VOWEL_LETTERS:
                return letter
        return None

    def front(index):
        v = colouring_vowel(index)
        return v is not None and v in FRONT_VOWEL_LETTERS

    plain = {
        "a": "a", "â": "a", "e": "e", "ı": "ɯ", "i": "i", "î": "i", "o": "o",
        "ö": "ø", "u": "u", "û": "u", "ü": "y", "c": "dʒ", "ç": "tʃ",
        "j": "ʒ", "ş": "ʃ", "y": "j", "q": "k", "w": "v", "x": "ks",
    }  # fmt: skip
    out = []
    for index, letter in enumerate(letters):
        last = index + 1 == len(letters)
        if letter in plain:
            out.append(plain[letter])
        elif letter == "r":
            out.append("r" if last else "ɾ")
        elif letter == "g":
            out.append("ɟ" if front(index) else "ɡ")
        elif letter == "l":
            out.append("l" if front(index) else "ɫ")
        elif letter == "ğ":
            previous = next((l for l in reversed(letters[:index]) if l in VOWEL_LETTERS), None)
            out.append("j" if previous is not None and previous in FRONT_VOWEL_LETTERS else "ː")
        else:
            out.append(letter)
    return _stress(word, "".join(out))


def phonemize_clause(clause, max_letters, label):
    ws = words(clause, max_letters)
    if not ws:
        return ""
    labels = label(ws)
    if len(labels) != len(ws):
        raise RuntimeError(f"the model answered {len(labels)} words for {len(ws)}")
    return " ".join(
        to_voice_symbols(w, l) if labels_line_up(w, l) else rule_phonemes(w)
        for w, l in zip(ws, labels)
    )


# ---- clauses and sentences (phonemes.rs) --------------------------------


def split_clauses(text):
    """[(clause text, terminator or None)], split at `, : ; . ! ?` followed by
    whitespace or the end of the text."""
    clauses, current = [], []
    for index, ch in enumerate(text):
        boundary = ch in CLAUSE_MARKS and (
            index + 1 == len(text) or text[index + 1].isspace()
        )
        if boundary:
            clause = "".join(current).strip()
            current = []
            if clause:
                clauses.append((clause, ch))
        else:
            current.append(ch)
    rest = "".join(current).strip()
    if rest:
        clauses.append((rest, None))
    return clauses


def sentence_phonemes(clauses, ipa):
    sentences, current = [], ""
    for (_, mark), raw in zip(clauses, ipa):
        current += raw
        if mark is not None:
            current += mark
            if mark in SPACED_MARKS:
                current += " "
        if mark is None or mark in SENTENCE_MARKS:
            sentences.append(current)
            current = ""
    if current:
        sentences.append(current)
    result = [list(unicodedata.normalize("NFD", s)) for s in sentences]
    return [s for s in result if s]


# ---- the model ----------------------------------------------------------


class DizgePhonemizer:
    def __init__(self, model_path, vocab_path, intra_threads=4):
        import onnxruntime as ort

        with open(vocab_path, encoding="utf-8") as f:
            vocab = json.load(f)
        self.pad, self.unk = vocab["pad"], vocab["unk"]
        self.cls, self.sep = vocab["cls"], vocab["sep"]
        self.max_letters = vocab["max_length"] - 2
        self.chars = vocab["chars"]
        self.labels = vocab["labels"]
        options = ort.SessionOptions()
        options.intra_op_num_threads = intra_threads
        self.session = ort.InferenceSession(
            model_path, options, providers=["CPUExecutionProvider"]
        )

    def _label(self, ws):
        import numpy as np

        width = max(min(len(w), self.max_letters) for w in ws) + 2
        ids = np.full((len(ws), width), self.pad, dtype=np.int64)
        mask = np.zeros((len(ws), width), dtype=np.int64)
        for row, w in enumerate(ws):
            tokens = (
                [self.cls]
                + [self.chars.get(ch, self.unk) for ch in w[: self.max_letters]]
                + [self.sep]
            )
            ids[row, : len(tokens)] = tokens
            mask[row, : len(tokens)] = 1
        logits = self.session.run(None, {"input_ids": ids, "attention_mask": mask})[0]
        best = logits.argmax(axis=-1)
        return [
            [self.labels[best[row, letter + 1]] for letter in range(min(len(w), self.max_letters))]
            for row, w in enumerate(ws)
        ]

    def phonemize_clause(self, clause):
        return phonemize_clause(clause, self.max_letters, self._label)

    def phonemize(self, voice, text):  # `voice` is ignored; same signature as eSpeak's
        clauses = split_clauses(text)
        ipa = [self.phonemize_clause(c) for c, _ in clauses]
        return sentence_phonemes(clauses, ipa)

In [ ]:
#@title 3. DizgeBERT modelini indir ve dene
import hashlib, sys, urllib.request
BASE = "https://raw.githubusercontent.com/ErdemGKSL/voice-me/main/piper-voices/g2p/tr-dizge"
os.makedirs("/content/dizge", exist_ok=True)
for name in ("model.onnx", "vocab.json", "SHA256SUMS"):
    urllib.request.urlretrieve(f"{BASE}/{name}", f"/content/dizge/{name}")
for line in open("/content/dizge/SHA256SUMS", encoding="utf-8"):
    digest, name = line.split()
    got = hashlib.sha256(open(f"/content/dizge/{name.lstrip('*')}", "rb").read()).hexdigest()
    assert got == digest, f"{name}: sha256 uyusmuyor ({got})"
print("SHA-256 doğru.")

sys.path.insert(0, "/content")
import importlib, dizge_phonemizer
importlib.reload(dizge_phonemizer)
dz = dizge_phonemizer.DizgePhonemizer("/content/dizge/model.onnx", "/content/dizge/vocab.json")
for t in ["Merhaba Erdem, güneş ve çikolata 12 kahvaltı olduğunu", "Bugün hava çok güzel. Yarın görüşürüz!"]:
    print(t, "->", ["".join(s) for s in dz.phonemize("tr", t)])

In [ ]:
#@title 4. Piper'a dizge yaması (tekrar çalıştırmak güvenli)
import pathlib
cfg = pathlib.Path(REPO_DIR, "src/piper/config.py")
ds = pathlib.Path(REPO_DIR, "src/piper/train/vits/dataset.py")

def patch(path, old, new):
    s = path.read_text(encoding="utf-8")
    if new in s:
        return
    assert s.count(old) == 1, f"{path.name}: yama noktası bulunamadı: {old!r}"
    path.write_text(s.replace(old, new), encoding="utf-8")

patch(cfg, '    TEXT = "text"\n', '    TEXT = "text"\n    DIZGE = "dizge"\n')
patch(ds, "import math\n", "import math\nimport os\nimport sys\n")
patch(ds, "phoneme_type=PhonemeType.ESPEAK,",
      'phoneme_type=PhonemeType(os.environ.get("PIPER_PHONEME_TYPE", "espeak")),')
patch(ds, "        phonemizer = EspeakPhonemizer()\n",
'''        if os.environ.get("PIPER_PHONEME_TYPE") == "dizge":
            sys.path.insert(0, "/content")
            from dizge_phonemizer import DizgePhonemizer
            phonemizer = DizgePhonemizer("/content/dizge/model.onnx", "/content/dizge/vocab.json")
        else:
            phonemizer = EspeakPhonemizer()
''')
# Eğitim düzeltmeleri (fork'ta eksikti):
# 1) eğitim verisi her epoch karıştırılsın, yarım kalan son batch atılsın;
lit = pathlib.Path(REPO_DIR, "src/piper/train/vits/lightning.py")
patch(ds, "            self.train_dataset,\n",
      "            self.train_dataset,\n            shuffle=True,\n            drop_last=True,\n")
# 2) öğrenme hızı zamanlayıcıları her epoch sonunda adımlansın (manual optimization
#    kullanıldığı için Lightning bunu kendisi yapmıyor, LR sabit kalıyordu);
patch(lit, "    def validation_step(self, batch: Batch, batch_idx: int):\n",
"""    def on_train_epoch_end(self):
        for scheduler in self.lr_schedulers():
            scheduler.step()

    def validation_step(self, batch: Batch, batch_idx: int):
""")
# 3) init_from_checkpoint (ince ayar) üreticiyle birlikte ayırt ediciyi de yüklesin.
patch(lit, "            self._load_generator_weights(init_from_checkpoint)\n",
      "            self._load_generator_weights(init_from_checkpoint)\n"
      "            self._load_discriminator_weights(init_from_checkpoint)\n")
patch(lit, "    def _load_generator_weights(self, ckpt_path: str):\n",
"""    def _load_discriminator_weights(self, ckpt_path: str):
        if not os.path.isfile(ckpt_path):
            return
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt.get("state_dict", ckpt)
        d_state = {k[len("model_d."):]: v for k, v in state.items() if k.startswith("model_d.")}
        if not d_state:
            _LOGGER.warning("init_from_checkpoint: no discriminator weights in %s", ckpt_path)
            return
        missing, _unexpected = self.model_d.load_state_dict(d_state, strict=False)
        _LOGGER.info("init_from_checkpoint: discriminator loaded (%d tensors, %d missing)",
                     len(d_state), len(missing))

    def _load_generator_weights(self, ckpt_path: str):
""")
os.environ["PIPER_PHONEME_TYPE"] = "dizge"
!cd {REPO_DIR} && git diff --stat && python -c "import piper.config as c; print(list(c.PhonemeType))"

In [ ]:
#@title 5. Veri: voice-me-recorder ya da Higgs veri seti zip'i
#@markdown Boş bırakırsan yükleme penceresi açılır. Drive'daki zip için tam yolu yaz; birden çok zip için virgülle ayır.
ZIP_PATH = "" #@param {type:"string"}
#@markdown Birden çok zip (ör. iki PC'den gelen yedekler) yüklenirse hepsi birleştirilir.
#@markdown Higgs veri setinin zip'i (`metadata.csv` + `wavs/*.flac`, 24 kHz) de olur: sesler eğitim hızına (22050 Hz) çevrilir.
#@markdown Her klibin tepe seviyesi buna getirilir (voice-me-recorder da 0.89 kullanır); Higgs sesleri ~10 dB kısık geldiği için ana model ve ince ayar verisi aynı seviyede olur. 0 = dokunma.
NORMALIZE_PEAK = 0.89 #@param {type:"number"}
import csv, json, shutil, subprocess, wave, zipfile
import numpy as np
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

zips = [p.strip() for p in ZIP_PATH.split(",") if p.strip()]
if not zips:
    from google.colab import files
    zips = [os.path.join("/content", z) for z in files.upload().keys()]

shutil.rmtree("/content/dataset", ignore_errors=True)
os.makedirs("/content/dataset/wavs")
os.makedirs("/content/dataset/raw")
rows, manifest, sources = {}, {}, {}
for z in zips:
    with zipfile.ZipFile(z) as archive:
        names = set(archive.namelist())
        assert "metadata.csv" in names and any(n.startswith("wavs/") for n in names), f"{z}: veri seti zip'i değil"
        if "manifest.json" in names:
            manifest = json.loads(archive.read("manifest.json"))
        for line in archive.read("metadata.csv").decode("utf-8-sig").splitlines():
            if "|" in line:
                name, text = line.split("|", 1)
                # Recorder rows name the file ("x.wav"); Higgs rows give only the id.
                key = os.path.splitext(name.strip())[0]
                rows.setdefault(key, text.strip())
        for n in names:
            stem, ext = os.path.splitext(os.path.basename(n))
            if n.startswith("wavs/") and ext in (".wav", ".flac") and ".." not in n and stem not in sources:
                target = os.path.join("/content/dataset/raw", os.path.basename(n))
                with archive.open(n) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                sources[stem] = target

model_name = "erdem-tr-dizge"
if manifest.get("speaker"):
    model_name = "".join(c if c.isalnum() else "-" for c in manifest["speaker"].lower()).strip("-") + "-tr-dizge"
# Piper's medium/high models train at 22050 Hz (x-low at 16000); anything else is converted.
sample_rate = str(manifest.get("sample_rate", 22050))
if sample_rate not in ("16000", "22050"):
    sample_rate = "22050"

def normalize_peak(path):
    """Scale the clip so its loudest sample sits at NORMALIZE_PEAK."""
    if NORMALIZE_PEAK <= 0:
        return
    with wave.open(path) as w:
        params, audio = w.getparams(), np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    peak = int(np.abs(audio.astype(np.int32)).max()) if audio.size else 0
    if peak == 0:
        return
    scaled = np.clip(np.round(audio.astype(np.float64) * (NORMALIZE_PEAK * 32767 / peak)), -32768, 32767)
    with wave.open(path, "wb") as w:
        w.setparams(params)
        w.writeframes(scaled.astype(np.int16).tobytes())

def to_training_wav(stem):
    """Copy as is if already mono/16-bit at the training rate, else convert with ffmpeg;
    then bring the peak level to NORMALIZE_PEAK."""
    src, dst = sources[stem], f"/content/dataset/wavs/{stem}.wav"
    same = False
    if src.endswith(".wav"):
        with wave.open(src) as w:
            same = w.getnchannels() == 1 and w.getsampwidth() == 2 and w.getframerate() == int(sample_rate)
    if same:
        os.replace(src, dst)
    else:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", sample_rate,
                        "-sample_fmt", "s16", dst], check=True)
        os.remove(src)
    normalize_peak(dst)

keep = [n for n in rows if n in sources]
with ThreadPoolExecutor(os.cpu_count() or 4) as pool:
    list(pool.map(to_training_wav, keep))
shutil.rmtree("/content/dataset/raw", ignore_errors=True)
rows = {f"{n}.wav": rows[n] for n in keep}
with open("/content/dataset/metadata.csv", "w", encoding="utf-8") as f:
    f.writelines(f"{n}|{t}\n" for n, t in sorted(rows.items()))
total = 0.0
for n in rows:
    with wave.open(f"/content/dataset/wavs/{n}") as w:
        assert w.getnchannels() == 1 and w.getsampwidth() == 2 and w.getframerate() == int(sample_rate), f"{n}: mono/16-bit/{sample_rate} Hz değil"
        total += w.getnframes() / w.getframerate()
print(f"{len(rows)} cümle, {total/60:.1f} dk ses, model adı: {model_name}")
if total < 30 * 60:
    print("UYARI: 30 dakikadan az ses var; sıfırdan eğitim için 8 saat, finetune için en az birkaç yüz cümle önerilir.")

# Fonem denetimi: eğitime geçmeden önce haritada olmayan sembol var mı?
from piper.phoneme_ids import DEFAULT_PHONEME_ID_MAP
seen, unknown = Counter(), Counter()
for i, text in enumerate(rows.values()):
    for sentence in dz.phonemize("tr", text):
        for ch in sentence:
            seen[ch] += 1
            if ch not in DEFAULT_PHONEME_ID_MAP:
                unknown[ch] += 1
    if i < 3:
        print(text, "->", ["".join(s) for s in dz.phonemize("tr", text)])
print("farklı sembol:", len(seen), "| haritada olmayan:", dict(unknown))
assert not unknown, "phoneme_id_map'te olmayan semboller var, eğitime geçme"

output_dir = f"{OUTPUT_ROOT}/{model_name}"
os.makedirs(output_dir, exist_ok=True)
audio_cache_dir = "/content/audio_cache_dizge"
shutil.rmtree(audio_cache_dir, ignore_errors=True)  # eski eSpeak fonemleri karışmasın
os.makedirs(audio_cache_dir)

In [ ]:
#@title 6. Eğitim (otomatik: kontrol noktası varsa devam eder)
#@markdown Boş: sıfırdan (Higgs 20k ana modeli böyle eğitilir). Doluysa (URL veya yol) bu kontrol noktasından finetune edilir: kendi sesinle ince ayar için ana modelin Drive'daki `.ckpt` yolunu yaz.
PRETRAINED_CHECKPOINT = "" #@param {type:"string"}
QUALITY = "medium" #@param ["x-low", "medium", "high"]
#@markdown T4 için 12, L4 için 24–32 (bellek hatası alırsan düşür).
BATCH_SIZE = 12 #@param {type:"integer"}
MAX_EPOCHS = 2000 #@param {type:"integer"}
#@markdown Numaralı kontrol noktası seyrek (her biri yüzlerce MB, disk dolmasın); `last.ckpt` sık yazılır.
CHECKPOINT_EVERY_EPOCHS = 50 #@param {type:"integer"}
LAST_CHECKPOINT_EVERY_EPOCHS = 2 #@param {type:"integer"}
#@markdown Öğrenme hızı her epoch bu oranla çarpılır: 0.9985 ile 1000. epoch'ta %22'sine, 1800'de %7'sine iner.
LR_DECAY = 0.9985 #@param {type:"number"}
#@markdown Kendi sesinle ince ayarda 32 daha kararlıdır; sıfırdan eğitimde 16-mixed hızlıdır.
PRECISION = "16-mixed" #@param ["16-mixed", "32"]
VALIDATION_FRACTION = 0.01 #@param {type:"number"}

import glob, re
os.environ["PIPER_PHONEME_TYPE"] = "dizge"
config_path = f"{output_dir}/config.json"
quality_args = {
    "x-low": "--model.hidden_channels 96 --model.inter_channels 96 --model.filter_channels 384 ",
    "medium": "",
    "high": ('--model.resblock 1 --model.resblock_kernel_sizes "(3, 7, 11)" '
             '--model.resblock_dilation_sizes "((1, 3, 5), (1, 3, 5), (1, 3, 5))" '
             '--model.upsample_rates "(8, 8, 2, 2)" --model.upsample_initial_channel 512 '
             '--model.upsample_kernel_sizes "(16, 16, 4, 4)" '),
}[QUALITY]

last = glob.glob(f"{output_dir}/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
extra = ""
if last:
    last.sort(key=lambda p: int((re.search(r"version_(\d+)", p) or [0, -1])[1]))
    extra = f'--ckpt_path "{last[-1]}" '
    print("Kaldığı yerden devam:", last[-1])
elif PRETRAINED_CHECKPOINT.strip():
    ckpt = PRETRAINED_CHECKPOINT.strip()
    if ckpt.startswith("http"):
        !aria2c --quiet=true --file-allocation=none -d /content -o pretrained.ckpt "{ckpt}"
        ckpt = "/content/pretrained.ckpt"
    assert os.path.exists(ckpt), "PRETRAINED_CHECKPOINT bulunamadı"
    extra = f'--model.init_from_checkpoint "{ckpt}" '
    print("Finetune:", ckpt)
else:
    print("Sıfırdan eğitim.")

val = (f"--data.validation_split {VALIDATION_FRACTION} --data.num_test_examples 1 "
       if VALIDATION_FRACTION > 0 else "--data.validation_split 0 --data.num_test_examples 0 ")
cmd = (
    f"cd {REPO_DIR} && PIPER_PHONEME_TYPE=dizge python -m piper.train fit "
    f'--data.voice_name "{model_name}" '
    '--data.csv_path "/content/dataset/metadata.csv" '
    '--data.audio_dir "/content/dataset/wavs" '
    '--data.espeak_voice "tr" '
    f'--data.cache_dir "{audio_cache_dir}" '
    f'--data.config_path "{config_path}" '
    f"--data.batch_size {BATCH_SIZE} --model.sample_rate {sample_rate} "
    + quality_args +
    f'--trainer.default_root_dir "{output_dir}" '
    "--trainer.accelerator gpu --trainer.devices 1 "
    f"--trainer.max_epochs {MAX_EPOCHS} --trainer.log_every_n_steps 1000 "
    f"--trainer.precision {PRECISION} " + val +
    f"--model.lr_decay {LR_DECAY} --model.lr_decay_d {LR_DECAY} "
    "--data.num_workers 4 --seed_everything 1234 "
    f"--checkpoint.every_n_epochs {CHECKPOINT_EVERY_EPOCHS} --checkpoint.save_top_k -1 "
    "--checkpoint.monitor null "
    f"--last_checkpoint.every_n_epochs {LAST_CHECKPOINT_EVERY_EPOCHS} " + extra
)
print(cmd)
get_ipython().system(cmd)

In [ ]:
#@title 7. ONNX export (voice-me için) ve indirme
import glob, json
ckpts = glob.glob(f"{output_dir}/lightning_logs/**/checkpoints/*.ckpt", recursive=True)
assert ckpts, "kontrol noktası yok; önce eğitim hücresini çalıştır"
latest = max(ckpts, key=os.path.getmtime)
print("kontrol noktası:", latest)
out_onnx = f"{output_dir}/{model_name}.onnx"
!cd {REPO_DIR} && python -m piper.train.export_onnx --checkpoint "{latest}" --output-file "{out_onnx}"
cfg_json = json.load(open(config_path, encoding="utf-8"))
assert cfg_json["phoneme_type"] == "dizge", cfg_json["phoneme_type"]
shutil.copy(config_path, out_onnx + ".json")
print("Hazır:", out_onnx, "ve", out_onnx + ".json", f"(phoneme_type=dizge, num_symbols={cfg_json['num_symbols']})")
bundle = f"/content/{model_name}-voice.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_onnx, f"{model_name}.onnx")
    z.write(out_onnx + ".json", f"{model_name}.onnx.json")
from google.colab import files
files.download(bundle)